# Introduction 

This notebook combines all the daily (cleaned) data from the `data_cleaning` notebook into a single data frame date will be used for the analysis. 

In [1]:
import pandas as pd
from datetime import datetime

# load filtered/imputed dataframes - step 2
path_base = "cache/step2/"
df_gas_no_outliers = pd.read_parquet(f"{path_base}df_gas_no_outliers.parquet")
df_elek_no_outliers = pd.read_parquet(f"{path_base}df_elek_no_outliers_good.parquet")
df_energy_out_no_outliers = pd.read_parquet(f"{path_base}df_energy_out_no_outliers.parquet")
df_outside_temp_imputed = pd.read_parquet(f"{path_base}df_outside_temp_imputed.parquet")

# load cached output of previous notebook
path_base = "cache/step1/"
df_supply_temp = pd.read_parquet(f"{path_base}df_supply_temp_active.parquet")
df_supply_temp = df_supply_temp.rename(columns={'supply_temp_active': 'supply_temp'})
df_return_temp = pd.read_parquet(f"{path_base}df_return_temp_active.parquet")
df_return_temp = df_return_temp.rename(columns={'return_temp_active': 'return_temp'})
df_flow = pd.read_parquet(f"{path_base}df_flow.parquet")
df_inside_temp = pd.read_parquet(f"{path_base}df_inside_temp.parquet")

# load from input
path_base = "input/"
df_house = pd.read_csv(f"{path_base}household.csv")
all_participant_ids = df_house["participant_id"].tolist()

In [2]:
from itertools import product

START_DATETIME = datetime.strptime("2022-01-01 00:00:00", "%Y-%m-%d %H:%M:%S")
END_DATETIME = datetime.strptime("2024-06-01 00:00:00", "%Y-%m-%d %H:%M:%S")

dates = pd.date_range(start=START_DATETIME, end=END_DATETIME)

# Use itertools.product to get all combinations of ids and dates
data = list(product(all_participant_ids, dates))

# Convert data into a DataFrame
df_merged = pd.DataFrame(data, columns=['participant_id', 'time'])
df_merged = df_merged.merge(df_gas_no_outliers[['time', 'gas_consumption', 'participant_id']], how='left', on=['time', 'participant_id'])
df_merged = df_merged.merge(df_elek_no_outliers[['time', 'participant_id', 'elek_consumption']], how='left', on=['time', 'participant_id'])
df_merged = df_merged.merge(df_supply_temp, how='left', on=['time', 'participant_id'])
df_merged = df_merged.merge(df_return_temp, how='left', on=['time', 'participant_id'])
df_merged = df_merged.merge(df_flow, how='left', on=['time', 'participant_id'])
df_merged = df_merged.merge(df_energy_out_no_outliers, how='left', on=['time', 'participant_id'])
df_merged = df_merged.merge(df_outside_temp_imputed, how='left', on=['time', 'participant_id'])
df_merged = df_merged.merge(df_inside_temp, how='left', on=['time', 'participant_id'])


# List of all column names
all_cols = df_merged.columns.tolist()

# Column names to exclude if these are na, we need at least some interesting value
cols_to_exclude = ['time', 'participant_id', 'outside_temp', 'inside_temp', 'is_outside_temp_imputed'] 

# Remove the specified column from the list
filtered_cols = [col for col in all_cols if col not in cols_to_exclude]
print(all_cols)
print(filtered_cols)
print(len(df_merged))
# filter only if selected cols are all NA
df_merged = df_merged.dropna(subset=filtered_cols, how='all')
print(len(df_merged))

['participant_id', 'time', 'gas_consumption', 'elek_consumption', 'supply_temp', 'return_temp', 'flowrate', 'energy_out', 'is_first_value', 'is_incomplete_yesterday', 'is_within_threshold', 'outside_temp', 'is_outside_temp_imputed', 'inside_temp']
['gas_consumption', 'elek_consumption', 'supply_temp', 'return_temp', 'flowrate', 'energy_out', 'is_first_value', 'is_incomplete_yesterday', 'is_within_threshold']
153642
92599


Compute three additional metrics: <br>
`delta_temp` = difference between outside and inside temp, indicating how big the heat demand from the house is.<br>
`energy_gas` = energy, in kWh, used by boiler burning gas (gas usage * 9.8 to convert m<sup>3</sup> gas to kWh, from `df_gas_no_outliers`)<br>
`energy_in` = energy used by heatpump (electricity from `df_electricity_no_outliers`) + energy of boiler <br>

In [3]:
df_merged['delta_temp'] = df_merged['inside_temp'] - df_merged['outside_temp']
df_merged['energy_gas'] = df_merged['gas_consumption'] * 9.77
df_merged['energy_in'] = df_merged['elek_consumption'] + df_merged['energy_gas']
df_merged = df_merged.sort_values(by=['participant_id', 'time']).reset_index(drop=True)
print(df_merged.dtypes)
df_merged

participant_id                     object
time                       datetime64[ns]
gas_consumption                   float64
elek_consumption                  float64
supply_temp                       float64
return_temp                       float64
flowrate                          float64
energy_out                        float64
is_first_value                     object
is_incomplete_yesterday            object
is_within_threshold                object
outside_temp                      float64
is_outside_temp_imputed            object
inside_temp                       float64
delta_temp                        float64
energy_gas                        float64
energy_in                         float64
dtype: object


,participant_id,time,gas_consumption,elek_consumption,supply_temp,return_temp,flowrate,energy_out,is_first_value,is_incomplete_yesterday,is_within_threshold,outside_temp,is_outside_temp_imputed,inside_temp,delta_temp,energy_gas,energy_in
0,-B51sQOp,2022-06-14,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,25.500000,NaN,NaN,NaN
1,-B51sQOp,2022-11-08,NaN,NaN,29.170000,23.640000,323.411765,NaN,NaN,NaN,NaN,13.700000,False,19.582000,5.882000,NaN,NaN
2,-B51sQOp,2022-11-09,NaN,NaN,29.757273,24.083182,347.645833,NaN,NaN,NaN,NaN,12.954167,False,19.853125,6.898958,NaN,NaN
3,-B51sQOp,2022-11-10,1.036,NaN,29.668158,24.814211,436.173913,38.78,False,False,True,10.854167,False,20.280220,9.426053,10.12172,NaN
4,-B51sQOp,2022-11-11,0.181,NaN,29.745000,25.128636,467.770833,41.81,False,False,True,9.520833,False,20.325000,10.804167,1.76837,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92594,zskLB4tU,2024-05-19,0.231,0.226,NaN,NaN,0.000000,0.00,False,False,True,NaN,NaN,20.629167,NaN,2.25687,2.48287
92595,zskLB4tU,2024-05-20,0.116,0.226,NaN,NaN,0.000000,0.00,False,False,True,NaN,NaN,20.244048,NaN,1.13332,1.35932
92596,zskLB4tU,2024-05-21,0.120,0.225,21.950000,19.760000,13.531250,0.00,False,False,True,NaN,NaN,20.077049,NaN,1.17240,1.39740
92597,zskLB4tU,2024-05-22,0.122,0.228,NaN,NaN,0.000000,0.00,False,False,True,NaN,NaN,19.990909,NaN,1.19194,1.41994


In [4]:
from pathlib import Path
Path("cache/step3/").mkdir(parents=True, exist_ok=True)

df_list = [
    (df_merged, "df_filtered_dataset"), 
]

for df_tuple in df_list: 
    df = df_tuple[0]
    name = df_tuple[1] 
    df.to_parquet(f"cache/step3/{name}.parquet", engine="pyarrow")
    print(f"Created .parquet file for {name}, length: {len(df)}") 

# Add a text file which notes when we last ran this code:
filepath = "cache/step3/notebook_3_last_run_date.txt"

with open(filepath, 'w') as file:
    file.write(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Created .parquet file for df_filtered_dataset, length: 92599
